# P4 — Are the risks right? Calibration, imbalance, thresholds, decision curves

AUROC (P3) only tells us whether the model **ranks** children correctly. At the bedside we need more:

| Question | Tool |
|---|---|
| When the model says "10% risk", do 10% of those children die? | **Calibration** |
| Should we "fix" the 4.4% imbalance by rebalancing? | **class weights / oversampling** experiment |
| At what risk do we raise an alert, and what does that cost? | **Threshold** → sensitivity, PPV, number needed to evaluate |
| Is using the model better than "treat everyone" or "treat no one"? | **Decision curve analysis** (net benefit) |

We keep the P3 model (11 predictors, logistic regression) and the bedside danger-sign score.

⚠️ Synthetic data — methods transfer, exact numbers may not.

## 1. Rebuild the P3 model and bedside score
Same predictors and pipeline as P3, wrapped in a function so we can create variants (class weights).

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import brier_score_loss, roc_auc_score, average_precision_score
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

X = pd.read_csv("p2_features.csv")
y = X.pop("inhospital_mortality").to_numpy()
yes = lambda col, val: (X[col] == val).astype(float).where(X[col].notna())
X["caprefill"] = yes("caprefill_adm", "Yes")
X["respdistress"] = yes("respdistress_adm", "Yes")
X["hiv_pos"] = yes("hivstatus_adm", "HIV positive")
X["malaria_pos"] = yes("malariastatuspos_adm", "Yes")
X["not_feeding"] = (X["feedingstatus_adm"] == "Not feeding at all").astype(float)

NUM = ["bcs_total", "waz", "lactate_mmolpl_adm", "spo2_min", "rr_ratio"]
BIN = ["lactate_measured", "caprefill", "respdistress", "hiv_pos", "malaria_pos", "not_feeding"]
PRED = NUM + BIN
bedside = (X["bcs_coma"] + X["sam"] + X["hypoxaemia"] + X["caprefill"].fillna(0)
           + X["respdistress"].fillna(0) + X["not_feeding"]).to_numpy()

def make_model(class_weight=None):
    prep = ColumnTransformer([
        ("num", Pipeline([("impute", SimpleImputer(strategy="median")), ("scale", StandardScaler())]), NUM),
        ("bin", SimpleImputer(strategy="most_frequent"), BIN)])
    return Pipeline([("prep", prep),
                     ("lr", LogisticRegression(C=1e6, max_iter=5000, class_weight=class_weight))])
print(len(PRED), "predictors,", y.sum(), "deaths")

## 2. Out-of-fold predictions: three versions of the same model
Every child gets a risk from a model that **never saw them** (5-fold CV). In each training fold we fit:

1. **plain**: the standard model.
2. **balanced**: `class_weight="balanced"`. Each death counts about 22× more than a survivor during fitting.
3. **oversampled**: deaths are copied (sampled with replacement) until they equal the survivors, **inside the training fold only**. Oversampling before splitting would put copies of test children into training, which is leakage.

Versions 2 and 3 are the usual "fixes" for imbalance you'll see in many ML papers.

We also choose an **alert threshold in each training fold**: the risk above which only 20% of *training* survivors fall (challenge rule: FPR ≤ 20%). We then apply it to the test fold. Choosing the threshold on the test data would be optimistic.

**What to look for:** the fold thresholds are around **4%**. That's close to the base rate, because the alert budget is generous.

In [ ]:
skf = StratifiedKFold(5, shuffle=True, random_state=1)
oof = {"plain": np.zeros(len(y)), "balanced": np.zeros(len(y)), "oversampled": np.zeros(len(y))}
flag = np.zeros(len(y), bool)
rng = np.random.default_rng(0)

for tr, te in skf.split(X, y):
    m = make_model().fit(X.iloc[tr][PRED], y[tr])
    oof["plain"][te] = m.predict_proba(X.iloc[te][PRED])[:, 1]

    p_tr = m.predict_proba(X.iloc[tr][PRED])[:, 1]
    thr = np.quantile(p_tr[y[tr] == 0], 0.80)          # FPR <= 20% on the training fold
    flag[te] = oof["plain"][te] >= thr
    print(f"fold threshold = {thr:.3f}")

    oof["balanced"][te] = make_model("balanced").fit(X.iloc[tr][PRED], y[tr]).predict_proba(X.iloc[te][PRED])[:, 1]

    pos = tr[y[tr] == 1]
    extra = rng.choice(pos, size=(y[tr] == 0).sum() - len(pos), replace=True)
    tr_os = np.concatenate([tr, extra])
    oof["oversampled"][te] = make_model().fit(X.iloc[tr_os][PRED], y[tr_os]).predict_proba(X.iloc[te][PRED])[:, 1]

## 3. Calibration: numbers first
| Measure | Meaning | Ideal |
|---|---|---|
| **Mean predicted vs observed** ("calibration-in-the-large") | Is the *average* risk right? | equal (O/E = 1) |
| **Calibration slope** | Are the predictions too extreme (< 1) or too timid (> 1)? | 1 |
| **Brier score** | Mean squared error of the probabilities (lower = better) | 0 |
| **Scaled Brier** | Brier compared to always predicting the prevalence (higher = better) | > 0 |

**What to look for:**
- **plain:** mean predicted 4.4% = observed 4.4% (O/E ≈ 1.0). The slope is ≈ 0.9: predictions are slightly too extreme, a mild sign of overfitting. Scaled Brier is positive but small (0.07). That's typical for a rare outcome and weak predictors.
- **balanced / oversampled:** the **AUROC is unchanged (≈ 0.75)**, but the mean predicted risk jumps to **~39%** against 4.4% observed. That's a 9-fold overestimate, and the Brier score is worse than predicting 4.4% for everyone (scaled Brier −3.2).

➡️ **Rebalancing does not help discrimination, and it destroys calibration** (van den Goorbergh et al., JAMIA 2022). A parent told "your child has a 40% risk of dying" when the true risk is 4% has been harmed by the model.

In [ ]:
def logit(p):
    p = np.clip(p, 1e-6, 1 - 1e-6)
    return np.log(p / (1 - p))

def calib_stats(y, p):
    lr = LogisticRegression(C=1e6).fit(logit(p).reshape(-1, 1), y)   # recalibration model
    brier = brier_score_loss(y, p)
    return {"mean predicted": p.mean(), "observed": y.mean(), "O/E": y.mean() / p.mean(),
            "slope": lr.coef_[0, 0], "Brier": brier,
            "scaled Brier": 1 - brier / (y.mean() * (1 - y.mean())),
            "AUROC": roc_auc_score(y, p), "AUPRC": average_precision_score(y, p)}

pd.DataFrame({k: calib_stats(y, p) for k, p in oof.items()}).round(3)

## 4. Calibration: the picture
**Reliability plot:** we sort children into 10 groups by predicted risk and plot the mean predicted risk against the observed death rate. On the diagonal = perfectly calibrated.

**What to look for:**
- **plain (left):** the points hug the diagonal. Most children sit below 6% predicted risk, and **almost all the signal is in the top decile** (~20% predicted, ~19% observed). It holds **50 of the 119 deaths**.
- **balanced (middle):** every point sits far to the right of the diagonal: the model *ranks* well, but the numbers are wrong.
- **Histogram (right):** the plain model gives most children a low risk. The balanced model spreads everyone upwards.

With only 119 deaths, each decile holds about 12 deaths. The wobble in the middle deciles is noise.

In [ ]:
def reliability(ax, p, title):
    dec = pd.qcut(p, 10, labels=False, duplicates="drop")
    pred = pd.Series(p).groupby(dec).mean(); obs = pd.Series(y).groupby(dec).mean()
    lim = max(pred.max(), obs.max()) * 1.1
    ax.plot([0, lim], [0, lim], c="grey", lw=.8)
    ax.plot(pred, obs, "o-")
    ax.set(xlabel="Mean predicted risk", ylabel="Observed mortality", title=title)

fig, ax = plt.subplots(1, 3, figsize=(15, 4.5))
reliability(ax[0], oof["plain"], "Plain model")
reliability(ax[1], oof["balanced"], "class_weight='balanced'")
ax[2].hist(oof["plain"], bins=50, alpha=.6, label="plain")
ax[2].hist(oof["balanced"], bins=50, alpha=.6, label="balanced")
ax[2].set(xlabel="Predicted risk", ylabel="Children", title="Distribution of predicted risk"); ax[2].legend()
plt.tight_layout(); plt.show()

dec = pd.qcut(oof["plain"], 10, labels=False)
pd.DataFrame({"mean predicted %": pd.Series(oof["plain"]).groupby(dec).mean() * 100,
              "observed %": pd.Series(y).groupby(dec).mean() * 100,
              "deaths": pd.Series(y).groupby(dec).sum()}).round(1)

## 5. The operating point: what happens on the ward
We apply the threshold learned on each training fold to its test fold. The result is a **confusion matrix**, the language of clinical tests:

| | Died | Survived |
|---|---|---|
| **Alert** | TP | FP (false alarm) |
| **No alert** | FN (missed death) | TN |

- **Sensitivity** = TP / all deaths
- **PPV** = TP / all alerts. **NNE** (number needed to evaluate) = 1 / PPV = how many children you assess per true death
- **NPV** = TN / all non-alerts

**What to look for:**
- FPR on the test folds ≈ **21%**, slightly above the 20% target. A threshold tuned on one dataset drifts on new data, so allow a margin.
- Sensitivity ≈ **60%**: 48 of 119 deaths are **missed**.
- PPV ≈ **12%**, so **NNE ≈ 8.5**: clinicians review about 8–9 alerted children per child who dies. That's acceptable for a cheap action (senior review, closer monitoring) and not for an expensive one (ICU transfer).
- NPV ≈ 97.7%: reassuring, but only because baseline mortality is 4.4%. "No alert" does not mean "safe".

In [ ]:
tp, fp = (flag & (y == 1)).sum(), (flag & (y == 0)).sum()
fn, tn = (~flag & (y == 1)).sum(), (~flag & (y == 0)).sum()
display(pd.DataFrame([[tp, fp], [fn, tn]], index=["Alert", "No alert"], columns=["Died", "Survived"]))

b = bedside >= 2
summary = pd.DataFrame({
    "model (train-fold threshold)": [tp / (tp + fn), fp / (fp + tn), tp / (tp + fp), (tp + fp) / tp, tn / (tn + fn), flag.mean()],
    "bedside score >= 2": [(b & (y == 1)).sum() / y.sum(), (b & (y == 0)).sum() / (y == 0).sum(),
                           (b & (y == 1)).sum() / b.sum(), b.sum() / (b & (y == 1)).sum(),
                           (~b & (y == 0)).sum() / (~b).sum(), b.mean()]},
    index=["sensitivity", "FPR", "PPV", "NNE", "NPV", "% children alerted"])
summary.round(3)

## 6. Decision curve analysis: is the model worth using at all?
AUROC ignores **consequences**. Decision curves ask: *if we act on the model, do we do more good than harm?*

A clinician chooses a **threshold probability** $p_t$: the risk at which they'd act. That choice encodes how they weigh harms:
- $p_t$ = 5% → "I'd accept 19 unnecessary escalations to catch 1 death" (odds 5:95 = 1:19)
- $p_t$ = 20% → "only 4 unnecessary escalations per death"

$$\text{Net benefit} = \frac{TP}{n} - \frac{FP}{n}\cdot\frac{p_t}{1-p_t}$$

= true deaths caught per child, minus false alarms weighted by their cost. We compare against two default strategies: **treat all** and **treat none** (net benefit 0).

**What to look for:**
- At very low thresholds (≤ 2%), the model ≈ treat-all: if you'd escalate everyone anyway, a model adds nothing.
- Between **~2.5% and 30%**, the model sits **above both defaults**: it's clinically useful across a sensible range.
- The **bedside rule** (≥ 2 signs, a single fixed operating point) matches the model around 5% and **falls below zero by ~13–15%**. A fixed rule can't adapt to different escalation costs; a calibrated probability can.
- Net benefit of 1.0 at $p_t$ = 10% means "1 extra death correctly identified per 100 children, with no extra unnecessary escalations", compared with treating none.

Decision curves **require calibrated probabilities**. The balanced model would look terrible here. Try it (exercise 2).

In [ ]:
def net_benefit(y, flagged, pt):
    n = len(y)
    return (flagged & (y == 1)).sum() / n - (flagged & (y == 0)).sum() / n * pt / (1 - pt)

pts = np.linspace(0.01, 0.30, 59)
nb = pd.DataFrame({
    "model": [net_benefit(y, oof["plain"] >= pt, pt) for pt in pts],
    "bedside score >= 2": [net_benefit(y, b, pt) for pt in pts],
    "treat all": [net_benefit(y, np.ones(len(y), bool), pt) for pt in pts],
    "treat none": 0.0}, index=pts) * 100

ax = nb.plot(figsize=(8, 5), style=["-", "-", "--", ":"])
ax.set_ylim(-1, y.mean() * 100 + .5)
ax.set(xlabel="Threshold probability p_t (risk at which you would escalate care)",
       ylabel="Net benefit (per 100 children)", title="Decision curve")
plt.tight_layout(); plt.show()
nb.loc[nb.index[[2, 8, 18, 28, 38]]].round(2)

## Summary
| Idea | Result here |
|---|---|
| Calibration of plain logistic model | Good: O/E 1.00, slope 0.90 (slightly overconfident) |
| Rebalancing (class weights, oversampling) | AUROC unchanged; predicted risk 39% vs observed 4.4% → **don't** |
| Threshold at FPR ≤ 20% (set on training folds) | Sensitivity 0.60, PPV 0.12 (NNE 8.5), test FPR 0.21 |
| Decision curve | Model beats treat-all/none from ~2.5% to 30%; bedside rule only near 5% |

**Key messages**
1. **Discrimination ≠ calibration.** A model can rank perfectly and still give wrong numbers.
2. **Don't rebalance** a clinical risk model. If you need more alerts, **lower the threshold** instead.
3. **Choose thresholds on training data**, report the resulting sensitivity, PPV and NNE, and expect drift.
4. **Net benefit** links the model to a clinical decision and its costs. Report it alongside AUROC (TRIPOD+AI).
5. With 4.4% mortality, even a decent model gives **PPV ≈ 12%**. Alerts should trigger cheap, safe actions.

## ✍️ Try it yourself
1. **Stricter alerts:** change `0.80` to `0.90` in §2 (FPR ≤ 10%). What happens to sensitivity, PPV and NNE?
2. **Decision curve of a badly calibrated model:** in §6 add a `"balanced"` column using `oof["balanced"] >= pt`. Why does it collapse, even though its AUROC is the same?
3. **Recalibration:** fit `LogisticRegression().fit(logit(oof["balanced"]).reshape(-1, 1), y)` and use its predictions. Does calibration recover? (This is how you'd rescue a miscalibrated model, and in real life it needs new data.)
4. **Clinical judgement:** in your PICU, which action would you attach to an alert, and what $p_t$ does that imply?

**Next — P5:** gradient boosting (can non-linear models beat logistic regression here?), SHAP explanations, and bootstrap confidence intervals.